<a href="https://colab.research.google.com/github/coreprimejio/ev-server/blob/master-qa/Quiz_PDF_Generator_Fractions_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import math
import random
import uuid
import shutil
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from pylatex import Document, Section, Command, Figure, MiniPage, LineBreak, Enumerate
from pylatex.utils import NoEscape, bold

# --- FIGURE GENERATION FUNCTIONS ---

def create_circular_division(total_parts, shaded_parts, output_filename):
    """Generates pie charts for circular divisions."""
    if total_parts <= 0: return

    num_wholes = shaded_parts // total_parts
    remaining_shaded = shaded_parts % total_parts

    num_charts = num_wholes + (1 if remaining_shaded > 0 else 0)
    if num_charts == 0: return

    fig, axes = plt.subplots(1, num_charts, figsize=(2.5 * num_charts, 2.5))
    if num_charts == 1: axes = [axes]

    for i in range(num_wholes):
        axes[i].pie([1], colors=['royalblue'], wedgeprops={'edgecolor': 'black', 'linewidth': 1})
        axes[i].axis('equal')

    if remaining_shaded > 0:
        sizes = [1] * total_parts
        colors = ['royalblue'] * remaining_shaded + ['lightgrey'] * (total_parts - remaining_shaded)
        axes[num_wholes].pie(sizes, colors=colors, startangle=90, counterclock=False,
                            wedgeprops={'edgecolor': 'black', 'linewidth': 1})
        axes[num_wholes].axis('equal')

    plt.savefig(output_filename, format='jpg', bbox_inches='tight', dpi=100)
    plt.close()

def create_rectangular_division(total_parts, shaded_parts, shading_style, output_filename):
    """Generates a horizontally long grid for rectangular divisions, handling improper fractions."""
    if total_parts == 0: return

    num_wholes = shaded_parts // total_parts
    remaining_shaded = shaded_parts % total_parts
    num_charts = num_wholes + (1 if remaining_shaded > 0 else 0)
    if num_charts == 0: return

    rows, cols = 1, total_parts
    if total_parts > 3:
        factors = []
        for i in range(1, int(math.sqrt(total_parts)) + 1):
            if total_parts % i == 0:
                factors.append((i, total_parts // i))
        rows, cols = min(factors, key=lambda x: abs(x[0] - x[1]))
        if rows > cols: rows, cols = cols, rows

    fig, axes = plt.subplots(1, num_charts, figsize=(cols * 0.5 * num_charts, rows * 0.5))
    if num_charts == 1: axes = [axes]

    for i in range(num_wholes):
        for j in range(total_parts):
            r, c = divmod(j, cols)
            axes[i].add_patch(Rectangle((c, rows - 1 - r), 1, 1, facecolor='royalblue', edgecolor='black', linewidth=1))
        axes[i].set_xlim(0, cols); axes[i].set_ylim(0, rows); axes[i].set_aspect('equal', 'box'); axes[i].axis('off')
        # Add a thick outer border
        axes[i].add_patch(Rectangle((0, 0), cols, rows, facecolor='none', edgecolor='black', linewidth=3))


    if remaining_shaded > 0:
        ax_frac = axes[num_wholes]
        indices = list(range(total_parts))
        shaded_indices = random.sample(indices, remaining_shaded)
        for j in range(total_parts):
            r, c = divmod(j, cols)
            is_shaded_part = j in shaded_indices
            if shading_style == 'half' and is_shaded_part:
                half1 = Rectangle((c, rows - 1 - r), 0.5, 1, facecolor='royalblue', edgecolor='black', linewidth=1)
                half2 = Rectangle((c + 0.5, rows - 1 - r), 0.5, 1, facecolor='lightgrey', edgecolor='black', linewidth=1)
                ax_frac.add_patch(half1)
                ax_frac.add_patch(half2)
            else:
                color = 'royalblue' if is_shaded_part else 'lightgrey'
                rect = Rectangle((c, rows - 1 - r), 1, 1, facecolor=color, edgecolor='black', linewidth=1)
                ax_frac.add_patch(rect)
        ax_frac.set_xlim(0, cols); ax_frac.set_ylim(0, rows); ax_frac.set_aspect('equal', 'box'); ax_frac.axis('off')
        # Add a thick outer border
        ax_frac.add_patch(Rectangle((0, 0), cols, rows, facecolor='none', edgecolor='black', linewidth=3))


    plt.savefig(output_filename, format='jpg', bbox_inches='tight', dpi=100)
    plt.close()

def create_triangular_division(total_parts, shaded_parts, shading_style, output_filename):
    """Generates triangular division figures, handling improper fractions."""
    divisions = int(math.sqrt(total_parts))
    if divisions * divisions != total_parts: return

    num_wholes = shaded_parts // total_parts
    remaining_shaded = shaded_parts % total_parts
    num_charts = num_wholes + (1 if remaining_shaded > 0 else 0)
    if num_charts == 0: return

    fig, axes = plt.subplots(1, num_charts, figsize=(4 * num_charts, 4))
    if num_charts == 1: axes = [axes]

    def draw_single_triangle(ax, is_full, shaded_count):
        side_length = 1.0; height = side_length * math.sqrt(3) / 2.0; all_triangles = []
        points = {}
        for row in range(divisions + 1):
            for col in range(row + 1):
                x = (col * side_length) - (row * side_length / 2.0); y = -row * height
                points[(row, col)] = (x, y)
        for row in range(divisions):
            for col in range(row + 1):
                p1, p2, p3 = points[(row, col)], points[(row + 1, col)], points[(row + 1, col + 1)]
                all_triangles.append([p1, p3, p2])
                if col < row:
                    p1_up, p2_up, p3_up = points[(row, col)], points[(row, col + 1)], points[(row + 1, col + 1)]
                    all_triangles.append([p1_up, p2_up, p3_up])

        shaded_indices = random.sample(range(total_parts), shaded_count) if not is_full else range(total_parts)
        for i, vertices in enumerate(all_triangles):
            is_shaded = i in shaded_indices
            if shading_style == 'half' and is_shaded:
                p1, p2, p3 = vertices; midpoint = ((p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2)
                ax.add_patch(Polygon([p1, midpoint, p3], facecolor='royalblue', edgecolor='black', linewidth=1))
                ax.add_patch(Polygon([midpoint, p2, p3], facecolor='lightgrey', edgecolor='black', linewidth=1))
            else:
                color = 'royalblue' if is_shaded else 'lightgrey'
                ax.add_patch(Polygon(vertices, facecolor=color, edgecolor='black', linewidth=1))
        ax.set_aspect('equal', 'box'); ax.axis('off')
        base_width, main_height = divisions * side_length, divisions * height
        ax.set_xlim(-base_width / 2 - 0.1, base_width / 2 + 0.1); ax.set_ylim(-main_height - 0.1, 0.1)

    for i in range(num_wholes):
        draw_single_triangle(axes[i], True, total_parts)
    if remaining_shaded > 0:
        draw_single_triangle(axes[num_wholes], False, remaining_shaded)

    plt.savefig(output_filename, format='jpg', bbox_inches='tight', pad_inches=0.1, dpi=100)
    plt.close()


# --- PDF GENERATION SCRIPT ---

def generate_quiz_pdf(quiz_data, pdf_filepath):
    """
    Reads a JSON object of questions and generates a single-column A4 PDF.
    """
    geometry_options = {"tmargin": "1in", "lmargin": "1in"}
    doc = Document(pdf_filepath, documentclass='article', document_options=['a4paper', '12pt'], geometry_options=geometry_options)

    doc.preamble.append(Command('usepackage', 'graphicx')); doc.preamble.append(Command('usepackage', 'amsmath'))
    doc.preamble.append(Command('usepackage', 'enumitem')); doc.preamble.append(Command('usepackage', 'xcolor'))
    doc.preamble.append(Command('setlength', [NoEscape(r'\parindent'), '0pt']))
    doc.preamble.append(NoEscape(r'\definecolor{questioncolor}{RGB}{40,80,150}'))

    # Dynamically create the title from the JSON data
    title = quiz_data.get('topic', 'Quiz')
    class_level = quiz_data.get('class', '')
    module_name = quiz_data.get('module', '')

    doc.append(NoEscape(r'\begin{center}'))
    doc.append(NoEscape(r'{\Huge\bfseries ' + title + r'}\\[5pt]'))
    doc.append(NoEscape(r'{\Large\bfseries ' + module_name + r'}\\[10pt]'))
    doc.append(NoEscape(r'{\large ' + class_level + r'}'))
    doc.append(NoEscape(r'\end{center}\bigskip'))

    output_pdf_dir = os.path.dirname(pdf_filepath)
    figures_dir = os.path.join(output_pdf_dir, 'figures')
    os.makedirs(figures_dir, exist_ok=True)

    for i, q in enumerate(quiz_data['questions']):
        doc.append(NoEscape(r'\textcolor{questioncolor}{\textbf{Question ' + str(i+1) + r':}} ')); doc.append(NoEscape(q['question']))
        doc.append(NoEscape(r'\par'))

        if 'figure_desc' in q:
            fig_desc = q['figure_desc']; fig_name_only = f'q_{i+1}.jpg'; fig_full_path = os.path.join(figures_dir, fig_name_only)

            if fig_desc['type_of_division'] == 'circular':
                create_circular_division(fig_desc['total_parts'], fig_desc['shaded_parts'], fig_full_path)
            elif fig_desc['type_of_division'] == 'rectangular':
                create_rectangular_division(fig_desc['total_parts'], fig_desc['shaded_parts'], fig_desc.get('shading_style', 'full'), fig_full_path)
            elif fig_desc['type_of_division'] == 'triangular':
                create_triangular_division(fig_desc['total_parts'], fig_desc['shaded_parts'], fig_desc['shading_style'], fig_full_path)

            if os.path.exists(fig_full_path):
                fig_relative_path = os.path.join('figures', fig_name_only)
                doc.append(NoEscape(r'\begin{center}')); doc.append(NoEscape(r'\includegraphics[width=0.3\linewidth]{' + fig_relative_path.replace('\\', '/') + '}')); doc.append(NoEscape(r'\end{center}'))

        with doc.create(Enumerate(options=NoEscape(r'label=(\alph*)'))) as enum:
            for option in q['options']:
                enum.add_item(NoEscape(option))

        doc.append(NoEscape(r'\bigskip\hrule\bigskip'))

    try:
        doc.generate_pdf(clean_tex=True)
        print(f"✅ Successfully generated PDF: {pdf_filepath}.pdf")
    except Exception as e:
        print(f"❌ PDF generation failed. Ensure a LaTeX distribution is installed. Error: {e}")

if __name__ == '__main__':
    input_dir = '/Users/ajaygupta/class5/fractions/json'
    output_dir = '/Users/ajaygupta/class5/fractions/pdfs'

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
        print(f"🧹 Cleaned up old directory: {output_dir}")

    os.makedirs(output_dir, exist_ok=True)

    json_file_path = os.path.join(input_dir, 'fractions.json')

    pdf_filename = f"fraction-{uuid.uuid4()}"
    pdf_full_path = os.path.join(output_dir, pdf_filename)

    try:
        with open(json_file_path, 'r') as f:
            quiz_json_object = json.load(f)
        generate_quiz_pdf(quiz_json_object, pdf_full_path)
    except FileNotFoundError:
        print(f"❌ Error: The file was not found at {json_file_path}")
    except json.JSONDecodeError:
        print(f"❌ Error: The file at {json_file_path} is not a valid JSON file.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

🧹 Cleaned up old directory: /Users/ajaygupta/class5/fractions/pdfs
✅ Successfully generated PDF: /Users/ajaygupta/class5/fractions/pdfs/fraction-27331bdb-1b8f-4381-a65d-b88572a3573d.pdf
